# 02 — Tensor Decompositions: SVD, CP, Tucker, Tensor Train

Decompositions break a large tensor into a product of smaller, simpler tensors.  
This is the foundation of compression, noise reduction, and tensor network design.

---

In [ ]:
import numpy as np
import tensorly as tl
from tensorly.decomposition import parafac, tucker, tensor_train
import matplotlib.pyplot as plt
from numpy.linalg import svd, norm

## 1. Singular Value Decomposition (SVD) — The Matrix Foundation

Every matrix $M \in \mathbb{R}^{m \times n}$ can be written as:

$$M = U \Sigma V^T$$

- $U \in \mathbb{R}^{m \times m}$: left singular vectors (orthogonal columns)
- $\Sigma \in \mathbb{R}^{m \times n}$: diagonal matrix of **singular values** $\sigma_1 \ge \sigma_2 \ge \cdots \ge 0$
- $V^T \in \mathbb{R}^{n \times n}$: right singular vectors (orthogonal rows)

**Truncated SVD** keeps only the top $r$ singular values — the best rank-$r$ approximation (Eckart-Young theorem):

$$M \approx \hat{M} = U_r \Sigma_r V_r^T$$

The error is $\|M - \hat{M}\|_F = \sqrt{\sigma_{r+1}^2 + \cdots + \sigma_{\min(m,n)}^2}$.

In [ ]:
np.random.seed(42)

# Create a rank-5 matrix with noise
m, n, true_rank = 50, 40, 5
M_clean = np.random.randn(m, true_rank) @ np.random.randn(true_rank, n)
M = M_clean + 0.1 * np.random.randn(m, n)

# Full SVD
U, sigma, Vt = svd(M, full_matrices=False)

print(f"M shape: {M.shape}")
print(f"U shape: {U.shape}")
print(f"sigma shape: {sigma.shape}")
print(f"Vt shape: {Vt.shape}")
print(f"\nTop-10 singular values: {sigma[:10].round(2)}")

# Plot singular values — the drop at rank 5 reveals the true rank
plt.figure(figsize=(7, 3))
plt.semilogy(sigma, 'o-', markersize=4)
plt.axvline(true_rank - 0.5, color='red', linestyle='--', label=f'True rank = {true_rank}')
plt.xlabel('Index'); plt.ylabel('Singular value (log)')
plt.title('Singular Value Spectrum'); plt.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Truncated SVD — best rank-r approximation
errors = []
for r in range(1, 20):
    M_approx = U[:, :r] @ np.diag(sigma[:r]) @ Vt[:r, :]
    err = norm(M - M_approx, 'fro') / norm(M, 'fro')
    errors.append(err)

plt.figure(figsize=(7, 3))
plt.plot(range(1, 20), errors, 'o-')
plt.axvline(true_rank - 0.5, color='red', linestyle='--', label=f'True rank = {true_rank}')
plt.xlabel('Rank r'); plt.ylabel('Relative Frobenius error')
plt.title('Truncated SVD Reconstruction Error'); plt.legend()
plt.tight_layout(); plt.show()

# Check at r=5
r = 5
M_r5 = U[:, :r] @ np.diag(sigma[:r]) @ Vt[:r, :]
print(f"Relative error at rank-{r}: {norm(M - M_r5,'fro') / norm(M,'fro'):.4f}")
print(f"Compression ratio: original {m*n} numbers -> {r*(m+1+n)} numbers")

## 2. CP Decomposition (CANDECOMP/PARAFAC)

The **CP decomposition** expresses a tensor as a sum of rank-1 terms:

$$\mathcal{T} \approx \sum_{r=1}^{R} \lambda_r \;\, \mathbf{a}_r^{(1)} \otimes \mathbf{a}_r^{(2)} \otimes \cdots \otimes \mathbf{a}_r^{(N)}$$

For a 3rd-order tensor:

$$T_{ijk} \approx \sum_{r=1}^{R} \lambda_r \, A_{ir} B_{jr} C_{kr}$$

- $R$ = **CP rank** (number of components)
- $A, B, C$ are **factor matrices** — columns are the component vectors
- Direct analogue of SVD for matrices

**Parameters**: $R \times (I + J + K)$ — linear in mode sizes, very compact.

In [ ]:
# Build a true rank-3 tensor and recover it with CP
np.random.seed(0)
I, J, K, R_true = 10, 8, 6, 3

# Ground truth factor matrices
A_true = np.random.randn(I, R_true)
B_true = np.random.randn(J, R_true)
C_true = np.random.randn(K, R_true)

T_true = np.einsum('ir,jr,kr->ijk', A_true, B_true, C_true)  # rank-R_true tensor
T_noisy = T_true + 0.05 * np.random.randn(I, J, K)

print(f"True tensor shape: {T_true.shape}")
print(f"True CP rank: {R_true}")

In [ ]:
# Fit CP decomposition with rank = 3
factors = parafac(tl.tensor(T_noisy), rank=3, n_iter_max=200, random_state=0)

# Reconstruct tensor from factors
T_recon = tl.cp_to_tensor(factors)

rel_err = norm(T_noisy - T_recon) / norm(T_noisy)
print(f"CP rank-3 relative error: {rel_err:.6f}")

# Factor matrices
weights, fac_matrices = factors
print(f"\nFactor matrix shapes: {[f.shape for f in fac_matrices]}")
print(f"Component weights (lambda): {weights.round(3)}")

# What happens if we use too few or too many components?
print("\nError vs CP rank:")
for r in [1, 2, 3, 4, 5]:
    f = parafac(tl.tensor(T_noisy), rank=r, n_iter_max=200, random_state=0)
    err = norm(T_noisy - tl.cp_to_tensor(f)) / norm(T_noisy)
    print(f"  rank={r}: err={err:.5f}")

## 3. Tucker Decomposition

Tucker is a **higher-order SVD (HOSVD)** — it generalises SVD to tensors:

$$\mathcal{T} \approx \mathcal{G} \times_1 U^{(1)} \times_2 U^{(2)} \times_3 U^{(3)}$$

- $\mathcal{G} \in \mathbb{R}^{R_1 \times R_2 \times R_3}$: **core tensor** (captures interaction between modes)
- $U^{(n)} \in \mathbb{R}^{I_n \times R_n}$: **factor matrices** (orthogonal, like U and V in SVD)
- $(R_1, R_2, R_3)$: **multilinear rank** — can compress each mode independently

**Parameters**: $R_1 R_2 R_3 + R_1 I_1 + R_2 I_2 + R_3 I_3$

Tucker is more flexible than CP but has a larger core. CP is a special case where $R_1=R_2=R_3=R$ and $\mathcal{G}$ is diagonal.

In [ ]:
np.random.seed(7)
T = np.random.randn(20, 15, 10)  # large tensor
print(f"Original tensor: shape={T.shape}, elements={T.size}")

# Tucker decomposition with rank (4, 3, 2)
ranks = (4, 3, 2)
core, factors = tucker(tl.tensor(T), rank=ranks)

print(f"\nCore tensor shape: {core.shape}")
print(f"Factor matrix shapes: {[f.shape for f in factors]}")

n_params_tucker = core.size + sum(f.size for f in factors)
print(f"\nOriginal params: {T.size}")
print(f"Tucker params:   {n_params_tucker}")
print(f"Compression:     {T.size / n_params_tucker:.1f}x")

# Reconstruct
T_recon = tl.tucker_to_tensor((core, factors))
rel_err = norm(T - T_recon) / norm(T)
print(f"\nRelative reconstruction error: {rel_err:.5f}")

In [ ]:
# How does Tucker rank affect error?
errors_tucker = []
rank_vals = list(range(1, 12))
for r in rank_vals:
    core_r, facs_r = tucker(tl.tensor(T), rank=(r, r, r))
    T_r = tl.tucker_to_tensor((core_r, facs_r))
    errors_tucker.append(norm(T - T_r) / norm(T))

plt.figure(figsize=(7, 3))
plt.plot(rank_vals, errors_tucker, 'o-')
plt.xlabel('Tucker rank (same for all modes)'); plt.ylabel('Relative error')
plt.title('Tucker Decomposition Error vs Rank')
plt.tight_layout(); plt.show()

## 4. Tensor Train (TT) Decomposition

The **Tensor Train** (a.k.a. Matrix Product State / MPS) expresses an order-$N$ tensor as a chain of 3-way tensors (cores):

$$T_{i_1 i_2 \cdots i_N} = G^{(1)}_{i_1} \, G^{(2)}_{i_2} \cdots G^{(N)}_{i_N}$$

Each $G^{(k)}$ is a 3-way tensor of shape $(r_{k-1}, i_k, r_k)$.  
The **bond dimensions** $r_0, r_1, \ldots, r_N$ (with $r_0 = r_N = 1$) control the approximation quality.

**Parameters**: $\sum_{k=1}^{N} r_{k-1} \cdot I_k \cdot r_k$ — exponential-to-linear reduction!

This is the core structure behind many quantum physics simulations and tensor network ML models.

In [ ]:
np.random.seed(3)
# 5th-order tensor — hard to store as dense
shape = (8, 7, 6, 5, 4)
T5 = np.random.randn(*shape)
print(f"Dense tensor: shape={T5.shape}, elements={T5.size}")

# TT decomposition with max rank 3
tt_cores = tensor_train(tl.tensor(T5), rank=[1, 3, 3, 3, 3, 1])

print("\nTT core shapes:")
for k, core in enumerate(tt_cores):
    print(f"  G^({k+1}): shape={core.shape}  (r_prev, i_k, r_next)")

n_params_tt = sum(c.size for c in tt_cores)
print(f"\nOriginal params: {T5.size}")
print(f"TT params:       {n_params_tt}")
print(f"Compression:     {T5.size / n_params_tt:.1f}x")

T5_recon = tl.tt_to_tensor(tt_cores)
print(f"Relative error:  {norm(T5 - T5_recon) / norm(T5):.5f}")

In [ ]:
# Visualise how bond dimension controls the trade-off
bond_dims = [1, 2, 3, 4, 5, 6, 8]
errors_tt, params_tt = [], []

for bd in bond_dims:
    ranks = [1] + [bd] * (len(shape) - 1) + [1]
    cores = tensor_train(tl.tensor(T5), rank=ranks)
    T_r = tl.tt_to_tensor(cores)
    errors_tt.append(norm(T5 - T_r) / norm(T5))
    params_tt.append(sum(c.size for c in cores))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3))
ax1.plot(bond_dims, errors_tt, 'o-'); ax1.set_xlabel('Bond dimension'); ax1.set_ylabel('Relative error'); ax1.set_title('TT Error vs Bond Dim')
ax2.plot(bond_dims, params_tt, 's-', color='orange'); ax2.axhline(T5.size, linestyle='--', color='gray', label='Dense'); ax2.set_xlabel('Bond dimension'); ax2.set_ylabel('# parameters'); ax2.set_title('TT Params vs Bond Dim'); ax2.legend()
plt.tight_layout(); plt.show()

## 5. Side-by-Side Comparison

| Decomposition | Parameters | Best for |
|--------------|------------|----------|
| Truncated SVD | $r(m+n)$ | Matrices, 2-way data |
| CP | $R(I+J+K)$ | Interpretable components, small rank |
| Tucker | $R_1 R_2 R_3 + \sum R_n I_n$ | Multi-modal compression, HOSVD |
| Tensor Train | $\sum r_{k-1} I_k r_k$ | High-order tensors, quantum systems |

In [ ]:
# Use the same 3-way tensor and compare all three at similar compression
np.random.seed(99)
I, J, K = 20, 18, 16
T = np.random.randn(I, J, K)
print(f"Test tensor shape: {T.shape}, original params: {T.size}\n")

# CP rank 5
f_cp   = parafac(tl.tensor(T), rank=5, n_iter_max=300, random_state=0)
T_cp   = tl.cp_to_tensor(f_cp)
p_cp   = sum(f.size for f in f_cp[1]) + f_cp[0].size

# Tucker (5,5,5)
core_t, facs_t = tucker(tl.tensor(T), rank=(5, 5, 5))
T_tuck = tl.tucker_to_tensor((core_t, facs_t))
p_tuck = core_t.size + sum(f.size for f in facs_t)

# TT bond dim 4
tt_c   = tensor_train(tl.tensor(T), rank=[1, 4, 4, 1])
T_tt   = tl.tt_to_tensor(tt_c)
p_tt   = sum(c.size for c in tt_c)

for name, T_r, p in [('CP   ', T_cp, p_cp), ('Tucker', T_tuck, p_tuck), ('TT   ', T_tt, p_tt)]:
    err = norm(T - T_r) / norm(T)
    print(f"{name}: params={p:4d} | compression={T.size/p:.1f}x | rel_err={err:.5f}")

## Summary

- **SVD** is the foundation — it gives the optimal low-rank matrix approximation.
- **CP** generalises SVD: sum of rank-1 tensors; very compact but NP-hard in general.
- **Tucker** is HOSVD: compress each mode independently via a core tensor.
- **Tensor Train** chains 3-way cores; ideal for high-order tensors and is numerically stable.

➡️ **03_tensor_network_structures.ipynb** — building and contracting network diagrams.